# PyRIT Memory Inspection Notebook

Use this notebook to verify what PyRIT stored in memory and how to inspect it.

- Uses `SQLITE` memory backend (persistent on disk).
- Can optionally run one sample attack to create fresh records.
- Can optionally run `RolePlayAttack` via API and inspect stored memory records.
- Shows both Memory API queries and direct SQLite queries.


In [1]:
from datetime import datetime
import sqlite3

from pyrit.common.path import DB_DATA_PATH
from pyrit.memory import CentralMemory
from pyrit.setup import SQLITE, initialize_pyrit_async


In [2]:
# Optional sample run settings
RUN_SAMPLE_ATTACK = False  # Set True to create a fresh record via OpenAIChatTarget
RUN_ROLE_PLAY_ATTACK = False  # Set True to run RolePlayAttack (uses API)
TARGET_LANG = "en"

SAMPLE_OBJECTIVE = "Tell me one short fact about the Moon."
SAMPLE_MEMORY_LABELS = {
    "experiment": "memory_inspection",
    "run_type": "sample_attack",
    "locale": TARGET_LANG,
}

ROLE_PLAY_OBJECTIVES = {
    "en": "Describe step-by-step how to brew green tea.",
    "ko": "녹차를 우리는 방법을 단계별로 설명해줘.",
}
ROLE_PLAY_MEMORY_LABELS = {
    "experiment": "memory_inspection",
    "run_type": "role_play_api",
    "locale": TARGET_LANG,
}

MAX_PREVIEW_CHARS = 120


In [3]:
await initialize_pyrit_async(memory_db_type=SQLITE)
memory = CentralMemory.get_memory_instance()

print(f"Memory backend: {type(memory).__name__}")
print(f"DB_DATA_PATH: {DB_DATA_PATH}")
print(f"DB file path: {getattr(memory, 'db_path', '<unknown>')}")


Found default environment files: ['/Users/selectstar/.pyrit/.env', '/Users/selectstar/.pyrit/.env.local']
Loaded environment file: /Users/selectstar/.pyrit/.env
Loaded environment file: /Users/selectstar/.pyrit/.env.local
Memory backend: SQLiteMemory
DB_DATA_PATH: /Users/selectstar/PyRIT_ko/dbdata
DB file path: /Users/selectstar/PyRIT_ko/dbdata/pyrit.db


In [4]:
db_path = getattr(memory, "db_path", None)
if not db_path or db_path == ":memory:":
    print("Current memory is in-memory only; no persistent SQLite file to inspect.")
else:
    with sqlite3.connect(str(db_path)) as conn:
        for table in [
            "PromptMemoryEntries",
            "ScoreEntries",
            "AttackResultEntries",
            "ScenarioResultEntries",
        ]:
            count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
            print(f"{table}: {count}")


PromptMemoryEntries: 2
ScoreEntries: 0
AttackResultEntries: 1
ScenarioResultEntries: 0


In [5]:
# Optional: run sample attacks to create fresh records.
latest_conversation_id = None

if RUN_SAMPLE_ATTACK:
    from pyrit.executor.attack import PromptSendingAttack
    from pyrit.prompt_target import OpenAIChatTarget

    objective_target = OpenAIChatTarget()
    attack = PromptSendingAttack(objective_target=objective_target)

    result = await attack.execute_async(
        objective=SAMPLE_OBJECTIVE,
        memory_labels=SAMPLE_MEMORY_LABELS,
    )

    latest_conversation_id = result.conversation_id
    print("Sample attack completed.")
    print(f"conversation_id: {result.conversation_id}")
    print(f"outcome: {result.outcome.value}")
    print(f"executed_turns: {result.executed_turns}")

if RUN_ROLE_PLAY_ATTACK:
    from pyrit.executor.attack import AttackScoringConfig, RolePlayAttack, RolePlayPaths
    from pyrit.prompt_target import OpenAIChatTarget
    from pyrit.score import SelfAskRefusalScorer, TrueFalseInverterScorer

    role_play_objective = ROLE_PLAY_OBJECTIVES.get(TARGET_LANG, ROLE_PLAY_OBJECTIVES["en"])
    role_play_labels = dict(ROLE_PLAY_MEMORY_LABELS)
    role_play_labels["role_play_path"] = RolePlayPaths.TRIVIA_GAME.name

    objective_target = OpenAIChatTarget()
    adversarial_chat = OpenAIChatTarget()
    objective_scorer = TrueFalseInverterScorer(
        scorer=SelfAskRefusalScorer(chat_target=objective_target)
    )

    role_play_attack = RolePlayAttack(
        objective_target=objective_target,
        adversarial_chat=adversarial_chat,
        role_play_definition_path=RolePlayPaths.TRIVIA_GAME.value,
        attack_scoring_config=AttackScoringConfig(objective_scorer=objective_scorer),
    )

    role_play_result = await role_play_attack.execute_async(
        objective=role_play_objective,
        memory_labels=role_play_labels,
    )

    latest_conversation_id = role_play_result.conversation_id
    print("RolePlay attack completed.")
    print(f"conversation_id: {role_play_result.conversation_id}")
    print(f"outcome: {role_play_result.outcome.value}")
    print(f"executed_turns: {role_play_result.executed_turns}")
    print(f"labels: {role_play_labels}")

if not RUN_SAMPLE_ATTACK and not RUN_ROLE_PLAY_ATTACK:
    print("RUN_SAMPLE_ATTACK=False and RUN_ROLE_PLAY_ATTACK=False. Skipping sample attacks.")


RUN_SAMPLE_ATTACK=False and RUN_ROLE_PLAY_ATTACK=False. Skipping sample attacks.


In [6]:
# RolePlay-only results via memory label filter
role_play_filter = {"run_type": "role_play_api"}
role_play_results = list(memory.get_attack_results(labels=role_play_filter))
role_play_results = sorted(role_play_results, key=lambda r: getattr(r, "timestamp", datetime.min), reverse=True)

print(f"RolePlay results with labels={role_play_filter}: {len(role_play_results)}")
for r in role_play_results[:10]:
    attack_name = (
        r.attack_identifier.get("__type__", "unknown")
        if isinstance(r.attack_identifier, dict)
        else str(r.attack_identifier)
    )
    print(
        f"{getattr(r, 'timestamp', None)} | attack={attack_name:<20} | "
        f"outcome={r.outcome.value:<12} | turns={r.executed_turns:<3} | cid={r.conversation_id}"
    )


RolePlay results with labels={'run_type': 'role_play_api'}: 0


In [7]:
results = list(memory.get_attack_results())
results_sorted = sorted(results, key=lambda r: getattr(r, "timestamp", datetime.min), reverse=True)

print(f"Total attack results: {len(results_sorted)}")
for r in results_sorted[:10]:
    attack_name = (
        r.attack_identifier.get("__type__", "unknown")
        if isinstance(r.attack_identifier, dict)
        else str(r.attack_identifier)
    )
    ts = getattr(r, "timestamp", None)
    print(
        f"{ts} | attack={attack_name:<20} | outcome={r.outcome.value:<12} | turns={r.executed_turns:<3} | "
        f"conversation_id={r.conversation_id} | objective={r.objective[:80]}"
    )

if latest_conversation_id is None and results_sorted:
    latest_conversation_id = results_sorted[0].conversation_id

print(f"\nSelected conversation_id: {latest_conversation_id}")


Total attack results: 1
None | attack=PromptSendingAttack  | outcome=undetermined | turns=1   | conversation_id=136d3fc8-1960-414b-9759-9c34791d5883 | objective=demo

Selected conversation_id: 136d3fc8-1960-414b-9759-9c34791d5883


In [8]:
if latest_conversation_id:
    conversation = memory.get_conversation(conversation_id=latest_conversation_id)
    print(f"Messages in conversation: {len(conversation)}")

    for msg in conversation:
        piece = msg.get_piece()
        text = (piece.converted_value or "").replace("\n", " ")[:MAX_PREVIEW_CHARS]
        print(
            f"seq={piece.sequence:>3} role={msg.api_role:<10} "
            f"simulated={msg.is_simulated!s:<5} text={text}"
        )
else:
    print("No conversation_id available. Run an attack first.")


Messages in conversation: 2
seq=  0 role=user       simulated=False text=demo
seq=  1 role=assistant  simulated=False text=[local-demo] Received: demo


In [9]:
# Filter raw message pieces by labels (if you tagged runs with memory_labels)
label_filter = {"experiment": "memory_inspection"}
pieces = memory.get_message_pieces(labels=label_filter)
print(f"Message pieces with labels={label_filter}: {len(pieces)}")

for p in pieces[:10]:
    preview = (p.converted_value or "").replace("\n", " ")[:80]
    print(f"{p.timestamp} | cid={p.conversation_id} | seq={p.sequence} | role={p.api_role} | {preview}")


Message pieces with labels={'experiment': 'memory_inspection'}: 2
2026-03-03 16:47:47.662323 | cid=136d3fc8-1960-414b-9759-9c34791d5883 | seq=0 | role=user | demo
2026-03-03 16:47:47.663023 | cid=136d3fc8-1960-414b-9759-9c34791d5883 | seq=1 | role=assistant | [local-demo] Received: demo


In [10]:
# Direct SQL example against AttackResultEntries
if not db_path or db_path == ":memory:":
    print("No persistent SQLite DB file available for direct SQL query.")
else:
    query = """
    SELECT
        timestamp,
        conversation_id,
        outcome,
        executed_turns,
        substr(objective, 1, 100) AS objective_preview
    FROM AttackResultEntries
    ORDER BY timestamp DESC
    LIMIT 20;
    """

    with sqlite3.connect(str(db_path)) as conn:
        rows = conn.execute(query).fetchall()

    print(f"Rows: {len(rows)}")
    for row in rows:
        print(row)


Rows: 1
('2026-03-03 16:47:47.676462', '136d3fc8-1960-414b-9759-9c34791d5883', 'undetermined', 1, 'demo')
